# NerGuard — Demo

**NerGuard** is an entropy-gated hybrid NER pipeline for privacy-compliant PII detection. It combines a multilingual mDeBERTa-v3 base model with optional LLM routing (OpenAI or Ollama) for uncertain spans.

- 📦 [PyPI](https://pypi.org/project/nerguard/) · 🤗 [Model on HuggingFace](https://huggingface.co/exdsgift/NerGuard-0.3B) · 💻 [GitHub](https://github.com/exdsgift/NerGuard)

**Runtime:** GPU recommended (T4 is fine). Go to **Runtime → Change runtime type → T4 GPU**.

## 1. Install

In [1]:
!pip install -q nerguard

## 2. Basic usage — NER model only (no API key needed)

The NER model (~300 MB) downloads automatically from HuggingFace on first use.

In [9]:
from nerguard import Redactor

ng = Redactor()

text = "Hi, I'm John Smith. My email is john.smith@acme.com and my SSN is 078-05-1120."
result = ng.redact(text)

print("Redacted:", result.text)
print("Mapping: ", result.mapping)
print("Entities:")
for e in result.entities:
    print(f"  [{e['label']}] '{e['text']}' (conf={e['confidence']:.3f}, source={e['source']})")

[NerGuard] Model not found locally — downloading from HuggingFace (exdsgift/NerGuard-0.3B)...
[NerGuard] Model not found locally — downloading from HuggingFace (exdsgift/NerGuard-0.3B)...
Redacted: Hi, I'm [NAME] [NAME]. My email is [EMAIL] and my SSN is [REFNUM].
Mapping:  {'NAME_0': 'John', 'NAME_1': 'Smith', 'EMAIL_0': 'john.smith@acme.com', 'REFNUM_0': '078-05-1120'}
Entities:
  [GIVENNAME] 'John' (conf=0.976, source=base model)
  [SURNAME] 'Smith' (conf=0.521, source=base model)
  [EMAIL] 'john.smith@acme.com' (conf=1.000, source=base + regex)
  [REFNUM] '078-05-1120' (conf=1.000, source=regex override)


## 3. Batch redaction

In [3]:
texts = [
    "Dear Maria García, your appointment is on 2024-03-15 at 10:30 AM.",
    "Please contact support at help@example.org or call +1-800-555-0199.",
    "Invoice for account IBAN DE89370400440532013000, tax ID 12-3456789.",
]

results = ng.redact_batch(texts)
for i, r in enumerate(results):
    print(f"[{i+1}] {r.text}")

[1] Dear [NAME] [NAME], your appointment is on [REFNUM] at [TIME].
[2] Please contact support at [EMAIL] or call +1-[REFNUM].
[3] Invoice for account IBAN [IBAN], tax ID [TAX].


## 4. LLM routing — higher accuracy on ambiguous spans (OpenAI)

LLM routing improves recall on uncertain predictions. To use it:

1. Open the 🔑 **Secrets** panel in the left sidebar (or go to **Tools → Secrets**).
2. Add a secret named `OPENAI_API_KEY` with your OpenAI key.
3. Enable **Notebook access** for that secret.

Then run the cell below.

In [6]:
from google.colab import userdata

openai_api_key = userdata.get("OPENAI_API_KEY")

ng_llm = Redactor(
    llm_routing=True,
    llm_source="openai",
    llm_model="gpt-4o",
    api_key=openai_api_key,
)

# Ambiguous text where LLM routing helps
text = "Call me at 078-05-1120. My card ending in 4111 1111 1111 1111 is active."
result = ng_llm.redact(text)

print("Redacted:", result.text)
print("Mapping: ", result.mapping)

ModuleNotFoundError: No module named 'google.colab'

## 5. Generic placeholders (maximum compression)

Use `typed=False` to replace all PII with `[PII]` instead of typed markers — useful when you want maximum token compression for downstream LLM context windows.

In [8]:
ng_generic = Redactor(typed=False)

text = "John Smith, born 1990-07-21, lives at 12 Baker Street, London."
result = ng_generic.redact(text)

print("Typed=False:", result.text)

[NerGuard] Model not found locally — downloading from HuggingFace (exdsgift/NerGuard-0.3B)...
[NerGuard] Model not found locally — downloading from HuggingFace (exdsgift/NerGuard-0.3B)...
Typed=False: [PII] [PII], born [PII], lives at [PII] [PII], [PII].


## 6. Interactive redaction

Enter any text below and redact it on the fly.

In [7]:
custom_text = "My name is Alice Dupont, I live in Paris and my phone is +33 6 12 34 56 78."  # @param {type:"string"}

result = ng.redact(custom_text)
print("Redacted:", result.text)
if result.entities:
    print("\nDetected entities:")
    for e in result.entities:
        print(f"  [{e['label']}] '{e['text']}' — conf={e['confidence']:.3f}")

Redacted: My name is [NAME] [NAME], I live in [CITY] and my phone is [PHONE].

Detected entities:
  [GIVENNAME] 'Alice' — conf=1.000
  [SURNAME] 'Dupont' — conf=0.994
  [CITY] 'Paris' — conf=0.999
  [TELEPHONENUM] '+33 6 12 34 56 78' — conf=1.000
